# 07.0. Pipeline: YOLO experiments

YOLO11s starting from the baseline, one change at a time, so the effect of the data and of the model is separate:
1. **Data:** train on frames with ignored regions masked (otherwise identical to the baseline).
2. **Model:** an extra high-resolution P2 detection level for tiny objects.
3. **Loss:** classification loss weight 1.0 instead of 0.5.

Sweep on fold 0, full cross-validation of the best, fold-ensemble submission.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import pandas as pd

from core import N_FOLDS
from core.cv import compare, cv_table, environment, run_cv
from core.data import assign_folds, find_data_root, load_pool, pool_targets
from core.fusion import fold_ensemble
from core.submission import write_submission
from core.yolo import P2_CFG, make_yolo_fns, plot_curves

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
targets = pool_targets(gt)
frames["fold"] = frames["video"].map(assign_folds(frames))
frames.groupby("fold").agg(clips=("video", "nunique"), frames=("uid", "size"))

## Experiments
Baseline settings: YOLO11s, COCO-pretrained, 1280 px, every 5th frame, 20 epochs, Ultralytics defaults, last epoch kept.
Each experiment changes one thing against `yolo_masked`, which differs from the baseline only by masking.

In [ ]:
BASE = dict(model="yolo11s.pt", imgsz=1280, epochs=20, batch=8, train_step=5, val_step=10,
            conf=0.001, iou=0.7, max_det=500, mask_ignore=True)
EXPERIMENTS = {
    "yolo_masked": BASE,
    "yolo_p2": {**BASE, "model_cfg": str(P2_CFG), "batch": 4},  # batch 4 so the extra level fits a T4
    "yolo_cls1": {**BASE, "train_args": {"cls": 1.0}},
}


def config(name):
    return {**EXPERIMENTS[name], "name": name}

## Sweep on fold 0

In [ ]:
for name in EXPERIMENTS:
    print(name)
    run_cv(frames, gt, config(name), *make_yolo_fns(config(name), WORK, targets), WORK, folds=[0])

### Fold-0 comparison
`baseline` vs `yolo_masked` is the effect of the data; the other rows are model and loss changes on masked data.

In [ ]:
sweep = compare([*EXPERIMENTS, "baseline"], WORK, fold=0)
sweep.round(4)

## Full cross-validation of the best experiment
Chosen on fold 0, so its fold-0 score is slightly optimistic.

In [ ]:
BEST = sweep.loc[sweep.index.isin(EXPERIMENTS), "mAP"].idxmax()
print("best on fold 0:", BEST)
_, predict_fn = make_yolo_fns(config(BEST), WORK, targets)
result = run_cv(frames, gt, config(BEST), *make_yolo_fns(config(BEST), WORK, targets), WORK)

In [ ]:
display(compare([BEST, "baseline"], WORK).round(4))  # out-of-fold over all frames
cv_table(result).round(4)

In [ ]:
plot_curves(BEST, N_FOLDS, WORK);

## Submission
The four fold models of the best experiment, fused by Weighted Boxes Fusion.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
test_dets = fold_ensemble(BEST, predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{BEST}.csv").head()